<a href="https://www.kaggle.com/code/emmglezz/mars-and-lunar-crater-detection?scriptVersionId=316923567" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Mars Crater Detection with YOLOv8

This project trains a computer vision model to automatically detect impact craters 
on the surface of Mars and the Moon using satellite imagery from NASA.

The dataset consists of labeled Mars and lunar surface images in YOLO format, 
sourced from the Martian/Lunar Crater Detection Dataset on Kaggle. I fine-tuned 
a YOLOv8 nano model on this data with the goal of deploying it as an interactive web app 
where anyone can upload a Mars image and get crater detections in real time.

**Model:** YOLOv8n (fine-tuned)  
**Dataset:** 143 labeled Mars/lunar images  
**Task:** Object detection — locating and bounding impact craters  
**Final mAP50:** 0.703

## Data Exploration

Before training, I loaded a sample image from the dataset and visualize it alongside 
its ground truth labels. Each crater is annotated in YOLO format: 
class, x_center, y_center, width, height — all normalized between 0 and 1 relative 
to image dimensions. I converted these to pixel coordinates and draw bounding boxes 
to verify the labels align correctly with the craters in the image.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg

train_path = '/kaggle/input/datasets/lincolnzh/martianlunar-crater-detection-dataset/craters/train'
img_path = train_path + '/images/011_png.rf.8ac312b4898f0106d10b76952a55d237.jpg'
img = mpimg.imread(img_path)

img_height, img_width = img.shape[:2]

fig, ax = plt.subplots()

ax.imshow(img)

with open(train_path + '/labels/011_png.rf.8ac312b4898f0106d10b76952a55d237.txt', 'r') as file:
    for line in file:
        arr = line.split()
        x_center = float(arr[1])
        y_center = float(arr[2])
        width = float(arr[3])
        height = float(arr[4])

        box_width = width * img_width
        box_height = height * img_height

        x_top_left = (x_center - width/2) * img_width
        y_top_left = (y_center - height/2) * img_height
        
        rect = patches.Rectangle((x_top_left, y_top_left), box_width, box_height, linewidth=1, edgecolor='red', fill=False)
        ax.add_patch(rect)

plt.show()

## Dataset Configuration

YOLOv8 requires a YAML configuration file that defines the paths to the train, validation, and test image folders, along with the number of classes and their names. Since the dataset didn't include one, I created it manually. This dataset has a single class: crater.

In [ ]:
import yaml

data = {
    'train': '/kaggle/input/datasets/lincolnzh/martianlunar-crater-detection-dataset/craters/train/images',
    'val': '/kaggle/input/datasets/lincolnzh/martianlunar-crater-detection-dataset/craters/valid/images',
    'test': '/kaggle/input/datasets/lincolnzh/martianlunar-crater-detection-dataset/craters/test/images',
    'nc': 1,
    'names': ['crater']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data, f)

## Model Training

I fine-tuned a pretrained YOLOv8 nano model on the crater dataset for 50 epochs at 640x640 resolution. Rather than training from scratch, fine-tuning starts from weights already learned on a large dataset, which works significantly better when the training data is small. The best weights are automatically saved during training based on validation performance.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640
)

## Inference on Test Image

Running the trained model on an unseen test image to verify it generalizes beyond the training data. The model outputs bounding boxes with confidence scores for each detected crater. Each detection shows the label "crater" followed by the confidence score, indicating how certain the model is about that particular detection.

In [ ]:
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

results = model('/kaggle/input/datasets/lincolnzh/martianlunar-crater-detection-dataset/craters/test/images/mars_crater--97-_jpg.rf.63347c2ec963cf5c4ab641e1ba872df1.jpg')

results[0].show()